# Example Agent: RSS News Reader

This notebook demonstrates how to build a custom agent using Kaval.AI's `Workflow` and the `python_tool` to read RSS feeds and have a conversation about the news.


## Building the Conversation Agent

Now we'll create a simple loop where the agent can discuss the fetched news with the user.
We'll use the workflow defined in `workflows/rss_agent.yaml`.


In [6]:
import asyncio
from kavalai.agents.workflow import Workflow

# Define YAML

rss_agent_yaml = """
name: News commentator bot.
description: Converses with the user on the topic of latest news available to it.
version: "1.0"
llm_model: openai/gpt-4o-mini
data_types:
  RssFeedItem:
    type: object
    properties:
      title: {type: string}
      link: {type: string}
      summary: {type: string}
  Feed:
    type: object
    properties:
      title: {type: string}
      url: {type: string}
      items: {type: array, items: {$ref: RssFeedItem}}
  hacker_news_feed: {$ref: Feed}
  antarctica_news_feed: {$ref: Feed}
  input:
    type: object
    properties:
      user_message: {type: string}
  output:
    type: object
    properties:
      agent_response: {type: string}
tasks:
  - name: Fetch Hacker news RSS
    type: python
    python_tool: kavalai.tools.rss.get_rss_feed
    inputs:
      url: {type: literal, value: "https://news.ycombinator.com/rss"}
      max_results: {type: literal, value: 15}
    output: hacker_news_feed
  - name: Fetch Antarctica news
    type: python
    python_tool: kavalai.tools.rss.get_rss_feed
    inputs:
      url: {type: literal, value: "https://feeds.feedburner.com/princess_elisabeth_station/news"}
      max_results: {type: literal, value: 15}
    output: antarctica_news_feed
  - name: discuss_news
    type: llm
    prompt: |
      Task: Summarize the latest news with two sentences less than 200 characters. Tell them to user and discuss the news with them.
    inputs:
      hacker_news_feed: {type: context}
      antarctica_news_feed: {type: context}
      user_message: {type: context, value: input.user_message}
    output: output
"""

workflow = Workflow.from_yaml(rss_agent_yaml)


## Run the Agent
Run the loop below to talk to the agent.

In [7]:
while True:
    user_input = input("You: ")
    if not user_input or user_input.lower() in ["exit", "quit", "stop"]:
        print("Agent: Goodbye!")
        break

    print(f"User: {user_input}")
    result = await workflow.run({"user_message": user_input})
    print(f"Agent: {result.data.agent_response}")




User: hi, how are you?
Agent: I'm doing well, thanks! Have you heard about the latest news? The Princess Elisabeth Antarctica station has officially closed for the season, and the team is heading home after a successful scientific season.
Agent: Goodbye!
